<div style="background:#0B3C49;color:white;padding:14px 18px;border-radius:8px">
<b style="color:#F2A541">INTERNATIONAL BALKAN UNIVERSITY · DS 501 INTRODUCTION TO DATA SCIENCE</b><br>
<span style="font-size:1.4em">Lab 4 · SQL for Data Science with Python</span><br>
Week 4 · Lessons 4-1 to 4-3 · about 75 minutes
</div>

## Learning goals
1. Connect to a relational database from Python and explore its schema.
2. Write SQL with `SELECT`, `WHERE`, `ORDER BY`, `JOIN`, `GROUP BY`, `HAVING`, subqueries and window functions.
3. Move query results into pandas for analysis and plotting.
4. Contrast the relational model with a document (NoSQL) representation.

**Database:** *Chinook*, a sample digital music store (customers, invoices, tracks, artists) used widely for teaching. It is a single SQLite file, so no server is needed.

In [ ]:
import sqlite3, urllib.request, os, json
import pandas as pd, matplotlib.pyplot as plt
DB = "chinook.sqlite"
if not os.path.exists(DB):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite", DB)
con = sqlite3.connect(DB)
def q(sql, **params):
    """Run a SQL query and return a DataFrame."""
    return pd.read_sql_query(sql, con, params=params or None)
q("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")

## 1. Explore the schema

In [ ]:
q("PRAGMA table_info(Invoice)")

In [ ]:
q("PRAGMA foreign_key_list(InvoiceLine)")

**✏️ Your turn (written).** Sketch the path of tables you must join to answer: *which music genre earns the most revenue?* Name the keys used at each step.

*Your answer:* 

## 2. Filtering and sorting

In [ ]:
q("""
SELECT FirstName, LastName, City, Country
FROM Customer
WHERE Country IN ('Germany', 'France')
ORDER BY Country, LastName
""")

**✏️ Your turn.** List the 10 longest tracks: track name, composer and length in **minutes** (rounded to 1 decimal), longest first.

In [ ]:
# Hint: Milliseconds / 60000.0, ORDER BY ... DESC, LIMIT 10
# Write your query, then remove the # signs:
# result = q("""
# SELECT ...
# """)
# result

## 3. Joins and aggregation

In [ ]:
revenue_by_country = q("""
SELECT c.Country, COUNT(DISTINCT c.CustomerId) AS customers, COUNT(i.InvoiceId) AS invoices,
       ROUND(SUM(i.Total), 2) AS revenue
FROM Customer c
JOIN Invoice i ON i.CustomerId = c.CustomerId
GROUP BY c.Country
ORDER BY revenue DESC
""")
revenue_by_country.head(10)

**✏️ Your turn.** Find **revenue by genre** (sum of UnitPrice × Quantity) for the top 8 genres, and draw a horizontal bar chart.

In [ ]:
# Hint: join InvoiceLine → Track → Genre, then GROUP BY genre
# Write your query, then remove the # signs:
# result = q("""
# SELECT ...
# """)
# result

**✏️ Your turn.** Which countries have **more than 5 customers**? Show the number of customers and the average invoice total. Use `HAVING`.

In [ ]:
# Write your query, then remove the # signs:
# result = q("""
# SELECT ...
# """)
# result

## 4. Subqueries and window functions

In [ ]:
# Customers who spent more than the average customer
q("""
SELECT CustomerId, ROUND(SUM(Total), 2) AS spent
FROM Invoice
GROUP BY CustomerId
HAVING SUM(Total) > (SELECT AVG(cust_total) FROM (SELECT SUM(Total) AS cust_total FROM Invoice GROUP BY CustomerId))
ORDER BY spent DESC
""").head()

In [ ]:
# Window function: rank customers by spending within their country
q("""
SELECT c.Country, c.FirstName || ' ' || c.LastName AS customer, ROUND(SUM(i.Total), 2) AS spent,
       RANK() OVER (PARTITION BY c.Country ORDER BY SUM(i.Total) DESC) AS rank_in_country
FROM Customer c JOIN Invoice i ON i.CustomerId = c.CustomerId
GROUP BY c.CustomerId
ORDER BY c.Country, rank_in_country
""").head(12)

**✏️ Your turn.** Using a window function, compute **monthly revenue** and a **running total** over time (use `strftime('%Y-%m', InvoiceDate)`). Plot the running total.

In [ ]:
# Hint: SUM(SUM(Total)) OVER (ORDER BY month)
# Write your query, then remove the # signs:
# result = q("""
# SELECT ...
# """)
# result

## 5. SQL or pandas?

In [ ]:
lines = q("""
SELECT il.InvoiceId, il.Quantity, il.UnitPrice, g.Name AS genre, mt.Name AS media
FROM InvoiceLine il JOIN Track t ON t.TrackId = il.TrackId
JOIN Genre g ON g.GenreId = t.GenreId JOIN MediaType mt ON mt.MediaTypeId = t.MediaTypeId
""")
lines.assign(revenue=lines.Quantity * lines.UnitPrice).pivot_table(index="genre", columns="media", values="revenue", aggfunc="sum", fill_value=0).round(2).head(8)

**✏️ Your turn (written).** When would you do the aggregation in SQL, and when in pandas?

*Your answer:* 

## 6. The same data as a document (NoSQL view)
A document database such as MongoDB would store one invoice with its lines nested inside it.

In [ ]:
inv = q("SELECT * FROM Invoice WHERE InvoiceId = 1").iloc[0]
items = q("SELECT t.Name AS track, il.UnitPrice, il.Quantity FROM InvoiceLine il JOIN Track t ON t.TrackId = il.TrackId WHERE il.InvoiceId = 1")
doc = {"invoice_id": int(inv.InvoiceId), "date": inv.InvoiceDate, "customer_id": int(inv.CustomerId),
       "billing_country": inv.BillingCountry, "total": float(inv.Total), "lines": items.to_dict(orient="records")}
print(json.dumps(doc, indent=1))

**✏️ Your turn (written).** Give one advantage and one disadvantage of storing invoices as documents instead of relational tables.

*Your answer:* 

In [ ]:
con.close()

---
### Submission checklist
- [ ] All ✏️ cells completed
- [ ] `Kernel → Restart & Run All` runs without errors
- [ ] Notebook saved and uploaded to Moodle before the next class